In [1]:
%load_ext autoreload
%autoreload 2
# %cd /mnt/sdd1/akanksha/formulacode/datasmith
%cd /mnt/sdd1/atharvas/formulacode/datasmith/

import datetime

import pandas as pd

from datasmith.logging_config import get_logger
from src.datasmith.execution.filter_commits import is_delinquient_repo

logger = get_logger("notebooks.building_reports_pr")

curr_date: str = datetime.datetime.now().isoformat()

/mnt/sdd1/atharvas/formulacode/datasmith


In [2]:
commits_df = pd.read_parquet(
    "/mnt/sdd1/atharvas/formulacode/datasmith/scratch/artifacts/pipeflush/merge_commits_filtered_with_patch.parquet"
)
for c in commits_df.columns.tolist():
    print(c)

sha
date
message
total_additions
total_deletions
total_files_changed
files_changed
patch
has_asv
file_change_summary
kind
repo_name
pr_url
pr_id
pr_node_id
pr_html_url
pr_diff_url
pr_patch_url
pr_issue_url
pr_number
pr_state
pr_locked
pr_title
pr_user
pr_body
pr_created_at
pr_updated_at
pr_closed_at
pr_merged_at
pr_merge_commit_sha
pr_assignee
pr_assignees
pr_requested_reviewers
pr_requested_teams
pr_labels
pr_milestone
pr_draft
pr_commits_url
pr_review_comments_url
pr_review_comment_url
pr_comments_url
pr_statuses_url
pr_head
pr_base
pr__links
pr_author_association
pr_auto_merge
pr_active_lock_reason
n_patch_tokens
total_changes
n_files_changed
is_perf
analysis_sha
analysis_repo_name
analysis_package_name
analysis_package_version
analysis_python_version
analysis_build_command
analysis_install_command
analysis_final_dependencies
analysis_can_install
analysis_dry_run_log
analysis_primary_root
analysis_resolution_strategy
pr_base_allow_forking
pr_base_archive_url
pr_base_archived
pr_base

In [3]:
print(commits_df.shape)
commits_df["is_delinquient_repo"] = commits_df["repo_name"].apply(is_delinquient_repo)
print(commits_df.query("~is_delinquient_repo").shape)
commits_df.groupby("repo_name").agg(
    num_prs=("pr_number", "nunique"),
    pr_stargazers=("pr_base_stargazers_count", "max"),
    pr_forks=("pr_base_forks_count", "max"),
    last_merge_date=("pr_merged_at", "max"),
).sort_values("num_prs", ascending=False)

(27166, 145)
(27166, 146)


,num_prs,pr_stargazers,pr_forks,last_merge_date
repo_name,,,,
pandas-dev/pandas,3232,46786,19104,2025-10-09T15:39:20Z
scikit-learn/scikit-learn,2483,63625,26313,2025-10-10T14:20:05Z
xdslproject/xdsl,2119,428,125,2025-10-10T09:09:33Z
apache/arrow,2090,16046,3874,2025-08-15T06:41:28Z
scipy/scipy,1475,14075,5494,2025-10-10T08:43:03Z
...,...,...,...,...
scikit-learn-contrib/metric-learn,5,1425,229,2019-03-14T10:19:57Z
spotify/voyager,5,1507,74,2024-12-19T07:44:26Z
jkjkil4/JAnim,3,188,14,2025-08-29T02:16:11Z


In [4]:
small_df = commits_df.query("pr_base_stargazers_count > 2000").sample(100)
for _, row in small_df.iterrows():
    print(row["pr_url"])

https://api.github.com/repos/napari/napari/pulls/4790
https://api.github.com/repos/pandas-dev/pandas/pulls/57965
https://api.github.com/repos/scipy/scipy/pulls/21131
https://api.github.com/repos/napari/napari/pulls/7741
https://api.github.com/repos/pandas-dev/pandas/pulls/51421
https://api.github.com/repos/apache/arrow/pulls/39125
https://api.github.com/repos/optuna/optuna/pulls/4433
https://api.github.com/repos/optuna/optuna/pulls/5671
https://api.github.com/repos/pandas-dev/pandas/pulls/48491
https://api.github.com/repos/optuna/optuna/pulls/4187
https://api.github.com/repos/scikit-image/scikit-image/pulls/5351
https://api.github.com/repos/pandas-dev/pandas/pulls/47233
https://api.github.com/repos/apache/arrow/pulls/37274
https://api.github.com/repos/holoviz/datashader/pulls/1405
https://api.github.com/repos/pymc-devs/pymc/pulls/5010
https://api.github.com/repos/holoviz/datashader/pulls/1378
https://api.github.com/repos/pandas-dev/pandas/pulls/46301
https://api.github.com/repos/scikit

In [11]:
# Updated to use new ReportBuilder interface
from datasmith.scrape.report_builder import ReportBuilder

# sample_pr = "https://api.github.com/repos/dedupeio/dedupe/pulls/997"
sample_pr = "https://api.github.com/repos/pandas-dev/pandas/pulls/54745"

# Initialize ReportBuilder with desired configuration
rb = ReportBuilder(
    enable_llm_backends=True,
    summarize_llm=False,
    add_classification=False,
    filter_performance_only=False,
    max_links_to_follow=60,
    model_name="@togetherai/meta-llama/Llama-3.3-70B-Instruct-Turbo",
    # model_name="@google/gemini-1.5-flash-latest",
)

# Build report using the pr_dict (row from dataframe)
# pr_dict = commits_df.query("pr_url == @sample_pr").iloc[0].to_dict()
# result = rb.build(pr_dict=pr_dict)

results = small_df.apply(rb.build, axis=1)

In [12]:
results_df = pd.json_normalize(results.apply(lambda d: d.__dict__))
small_df = pd.concat([small_df.reset_index(drop=True), results_df.reset_index(drop=True)], axis=1)

In [20]:
# results_df

In [ ]:
small_df.value_counts("is_performance_commit")

In [21]:
for row in small_df.to_dict(orient="records"):
    print(f"PR URL: {row['pr_url']}")
    # print(f"Classification: {row['classification']}")
    print("-----")
    print("REPORT")
    print(row["report_md"])
    print("-----")
    print("PROBLEM STATEMENT")
    print(row["problem_statement"])
    print("-----")
    print("HINTS")
    print(row["hints"])
    print("\n\n")

PR URL: https://api.github.com/repos/napari/napari/pulls/4790
-----
REPORT

### Hints

**brisvag** — 09:58 11/07/2022

@haesleinhuepf It would take something like the following to work with any n-d dataset:

```py
displayed_extrema = np.array(viewer.dims.range)[np.array(viewer.dims.displayed)]
displayed_ranges = displayed_extents[:, 1] - displayed_extents[:, 0]
points_layer.size = displayed_ranges * 0.01
```

However, for the reasons I stated [in this comment and previous ones](https://github.com/napari/napari/issues/4705#issuecomment-1160505164), I don't think that's something we should do; it relies too much on unrelated viewer state. I would rather add a special `'auto'` value that explicitly sets the size automatically, but I wouldn't make it default. Also, we still need to figure out how/if we can overcome the issues about instantiating layers before they are in the viewer (which happens with `viewer.add_points()`); are we leaving `auto` as `size` until rendering happens? And are 

/tmp/ipykernel_1130058/4291337756.py:1: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  for row in small_df.to_dict(orient="records"):


In [ ]:
from tqdm.auto import tqdm

from datasmith.scrape.build_pr_report import build_pr_report
from datasmith.scrape.report_builder import ReportBuilder

reports = []

# Initialize ReportBuilder once for all reports
rb = ReportBuilder(
    enable_llm_backends=False,
    summarize_llm=False,
    add_classification=True,  # Enable classification for batch processing
    max_links_to_follow=60,
)

# Process reports in batch
h_map = {}
for _, pr in tqdm(small_df.iterrows()):
    if not pr["pr_url"]:
        reports.append(None)
        continue

    print(pr["pr_url"])

    # Convert row to dict and build report
    pr_dict = pr.to_dict()
    result = rb.build(pr_dict=pr_dict)

    # Store results
    h_map[pr["pr_url"]] = (
        result.report_md,
        result.problem_statement,
        result.hints,
        result.classification,
        result.difficulty,
        result.is_performance_commit,
    )
    reports.append(result.report_md)
    break
report = h_map[next(iter(h_map))][0] if h_map else ""

In [ ]:
print(report)


### Hints



### Problem Statement


[ISSUE_NUM] 
I implement the `atcoder.math.pow_mod` as a wrapper of `pow` without introducing additional assertions.
Do we need to modify `docs/math.rst` either?



### Classification

13. Use a higher-level system that optimizes for you


### Difficulty

easy


In [ ]:
report, prob_statement, hints, classification, difficulty, performance_issue = build_pr_report(
    link="https://api.github.com/repos/xarray-contrib/xbatcher/pulls/112",
    summarize_llm=True,
    add_classification=False,
    patch="",
)
print(prob_statement)

## Problem
Generating batches in `__init__` is slow and memory intensive, related to [ISSUE_NUM] and [ISSUE_NUM]. Initialization is changed to load indices into memory rather than corresponding datasets. The change enabled the initialization of a 1.7 TB dataset with 1m+ samples, 30+ spatial features, in about 10 seconds which was previously overloading memory.

Rather than filling `_batches` with `DataArrays` and `Datasets`, it is filled with indices from `selector = {key: slice for key, slice in zip(dims, slices)}`. This required an update to the `concat_input_dims` option where the operation is done in `__getitem__()`. It is possible that this change decreases performance when this `concat_input_dims=True`.

## Related Issues
The issue is related to the following issues:
- Issue 0: Generate batches lazily
- Issue 1: Cache batches

### Issue 0: Generate batches lazily
This issue discusses the need to generate batches lazily to improve performance. A similar implementation was made in 

In [ ]:
from pprint import pprint

pprint(h_map)

{'https://api.github.com/repos/not522/ac-library-python/pulls/55': ('NOT_A_VALID_PR',
                                                                    '',
                                                                    '',
                                                                    '',
                                                                    '',
                                                                    False),
 'https://api.github.com/repos/not522/ac-library-python/pulls/56': ('NOT_A_VALID_PR',
                                                                    '',
                                                                    '',
                                                                    '',
                                                                    '',
                                                                    False),
 'https://api.github.com/repos/not522/ac-library-python/pulls/58': ('NOT_A_VALID_PR',
              

In [ ]:
h_map["https://api.github.com/repos/not522/ac-library-python/pulls/70"]

('\n### Hints\n\n\nThe code has been reviewed and appears to be good to go, with no issues or concerns raised.\n\n\n### Performance Issue\n\n{"label": "YES", "reason": "Implementing pow_mod as a wrapper of pow", "confidence": 80, "flags": ["mentions-speed", "startup"]}\n\n\n### LLM Generated summary\n\n### Problem\nThe issue is about implementing the `atcoder.math.pow_mod` function as a wrapper of the built-in `pow` function without introducing additional assertions.\n\n### Background\nThe discussion started with a comment on Issue 67, which suggested adding `pow_mod` in `math.py`. It was noted that Python\'s built-in `pow` function has a `mod` argument, and it was proposed to introduce `atcoder.math.pow_mod` as a wrapper for consistency with the original ACL.\n\n### Proposed Implementation\nThe user wants to implement the wrapper and has two questions:\n1. Do we need additional assertions in the wrapper?\n2. Do we still need to add tests for that?\n\n### Analysis\nThe original impleme

In [ ]:
count = sum(1 for v in h_map.values() if v[0] not in ("NOT_A_PERFORMANCE_COMMIT", "NOT_A_VALID_PR"))
print(count)

1


In [ ]:
# df = pd.DataFrame([{**c, **{"report": report}} for c, report in zip(commits, reports)])

In [ ]:
# df[["url", "report"]][~df["report"].str.contains("NOT_A_VALID_PR")].url.values

In [ ]:
# logger.info(f"PR Report:\n{report}")
print(report)

NOT_A_VALID_PR
